# Per-Subject DSAR Erasure · 3. Physical purge (CCPA "no trace")

The erasure UPDATE rewrote the live values, but Delta retains prior file versions for time-travel — the
original raw bytes are still on storage and recoverable. CCPA requires them **gone**. This notebook:

1. **Shortens history + VACUUMs** each erased table so the pre-erasure data files are physically deleted.
   - `REORG TABLE … APPLY (PURGE)` compacts and removes soft-deleted data (also purges deletion vectors).
   - Setting the table property **`delta.deletedFileRetentionDuration = interval 0 hours`** then running a
     plain `VACUUM` (no `RETAIN` clause) deletes the now-unreferenced old files immediately. We use the table
     property rather than the session conf `spark.databricks.delta.retentionDurationCheck.enabled`, because
     that session conf is **not settable on serverless / Spark Connect** — the property approach works
     everywhere and is scoped to just these tables.
2. **Scrubs the `dsar_request` table itself** — replaces the subject's raw first/last/email with the
   redaction token (so the request table keeps no trace either) and marks the request **COMPLETE**.

> ⚠️ Zero-retention VACUUM is destructive and disables time-travel recovery — that is the point for CCPA
> erasure. Run only after you've confirmed the erasure in `02` looks right (use its dry-run first).

## 0. Configuration

In [3]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema",  "allegiant_air_dsar", "2 Schema")
dbutils.widgets.text("redaction_token", "***REDACTED***", "3 Redaction token")
dbutils.widgets.dropdown("do_purge", "true", ["true","false"], "4 Actually VACUUM (false = REORG only)")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
TOKEN   = dbutils.widgets.get("redaction_token")
DO_PURGE = dbutils.widgets.get("do_purge") == "true"
FQ      = f"{CATALOG}.{SCHEMA}"
print("Schema:", FQ, "| do_purge:", DO_PURGE)

Schema: dkushari_uc.allegiant_air_dsar | do_purge: True

## 1. Tables to purge = every table in the registry

In [5]:
tables = [r["table_name"] for r in spark.sql(
    f"SELECT DISTINCT table_name FROM {FQ}.pii_column_registry").collect()]
print("will purge:", tables)

will purge: ['contact_email_only', 'loyalty_split', 'booking', 'customer_profile', 'app_hits_json']

## 2. REORG + VACUUM each table (physical removal of old bytes)
`purge_table()` sets `delta.deletedFileRetentionDuration = interval 0 hours` on the table, runs
`REORG … APPLY (PURGE)`, then a plain `VACUUM` (no RETAIN clause → uses the 0-hour property). This works on
serverless, unlike the `spark.databricks.delta.retentionDurationCheck.enabled` session conf.

In [7]:
def purge_table(fqt, do_vacuum=True):
    print(f"--- {fqt} ---")
    spark.sql(f"ALTER TABLE {fqt} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
    try:
        spark.sql(f"REORG TABLE {fqt} APPLY (PURGE)")
        print("  REORG APPLY(PURGE) done")
    except Exception as e:
        print("  REORG skipped:", str(e).splitlines()[0])
    if do_vacuum:
        spark.sql(f"VACUUM {fqt}")          # no RETAIN clause -> honours the 0-hour property
        print("  VACUUM done (0-hour retention)")
    else:
        print("  VACUUM skipped (do_purge=false)")

for t in tables:
    purge_table(f"{FQ}.{t}", DO_PURGE)

--- dkushari_uc.allegiant_air_dsar.contact_email_only ---
  REORG APPLY(PURGE) done
  VACUUM done (0-hour retention)
--- dkushari_uc.allegiant_air_dsar.loyalty_split ---
  REORG APPLY(PURGE) done
  VACUUM done (0-hour retention)
--- dkushari_uc.allegiant_air_dsar.booking ---
  REORG APPLY(PURGE) done
  VACUUM done (0-hour retention)
--- dkushari_uc.allegiant_air_dsar.customer_profile ---
  REORG APPLY(PURGE) done
  VACUUM done (0-hour retention)
--- dkushari_uc.allegiant_air_dsar.app_hits_json ---
  REORG APPLY(PURGE) done
  VACUUM done (0-hour retention)

## 3. Scrub the request table + mark COMPLETE
Remove the raw identifiers from `dsar_request` (redaction token) and set `status='COMPLETE'` for the
requests we just processed. This mirrors the current flow, which also erases the request record's own values.

In [9]:
def sqlstr(v):
    return "'" + v.replace("'", "''") + "'"

# only scrub PENDING (just-processed) rows; keep COMPLETE ones as-is
spark.sql(f"""
UPDATE {FQ}.dsar_request
SET first_name = {sqlstr(TOKEN)},
    last_name  = {sqlstr(TOKEN)},
    email      = {sqlstr(TOKEN)},
    status     = 'COMPLETE'
WHERE status = 'PENDING'
""")
# purge its history too
purge_table(f"{FQ}.dsar_request", DO_PURGE)
display(spark.table(f"{FQ}.dsar_request"))

request_id,first_name,last_name,email,request_type,request_date,deadline_date,status
REQ-0001,***REDACTED***,***REDACTED***,***REDACTED***,DELETE,2026-07-23,2026-09-06,COMPLETE
REQ-0002,***REDACTED***,***REDACTED***,***REDACTED***,OBFUSCATE,2026-07-23,2026-09-06,COMPLETE
REQ-0003,***REDACTED***,***REDACTED***,***REDACTED***,DELETE,2026-07-23,2026-09-06,COMPLETE
REQ-0004,***REDACTED***,***REDACTED***,***REDACTED***,OBFUSCATE,2026-07-23,2026-09-06,COMPLETE
REQ-0005,***REDACTED***,***REDACTED***,***REDACTED***,DELETE,2026-07-23,2026-09-06,COMPLETE
REQ-0006,***REDACTED***,***REDACTED***,***REDACTED***,DELETE,2026-07-23,2026-09-06,COMPLETE
REQ-0007,***REDACTED***,***REDACTED***,***REDACTED***,DELETE,2026-07-23,2026-09-06,COMPLETE
REQ-0008,***REDACTED***,***REDACTED***,***REDACTED***,OBFUSCATE,2026-07-23,2026-09-06,COMPLETE
REQ-0009,***REDACTED***,***REDACTED***,***REDACTED***,DELETE,2026-07-23,2026-09-06,COMPLETE
REQ-0010,***REDACTED***,***REDACTED***,***REDACTED***,DELETE,2026-07-23,2026-09-06,COMPLETE


## 4. Note
Raw bytes for the erased subjects are now gone (no time-travel recovery). Run **`04_orchestrate_and_validate`**
to confirm no trace remains and to see the whole flow wired as a monthly job.